# TOPIC MODELLING WITH NMF

In this notebook we are going to show how to do topic modelling with NMF (non-negative matrix factorization) on the emails of the debt collection agency. The goal is to learn how we can extract topics from natural language text using matrix factorization.

Note the difference with the transformers tutorial. There, every phone call had a label (how the call ended), and we trained a model to predict it, i.e. supervised learning. The emails don't have such a label: the `outcome` column only tells us if an email was sent, opened or replied to, not what the email is about. So here we let the data tell us which themes are in there, without any labels. This is unsupervised learning.

The idea is the following. We turn the emails into a big documents x words matrix, and then we use NMF to factorize it into two smaller matrices: one that tells us which words make up each topic, and one that tells us which topics each email is made of. We can then use the topics to answer questions like: what is the agency writing about? And what are the debtors writing back?

We will first look at the data, then clean and tokenize the emails, build the tf-idf matrix, and fit the NMF model. After that we check whether the topics make sense, look at how many topics we should use, and finally we zoom in on the replies of the debtors.

In the following, we will use both sklearn (for the NMF model) and the spacy NLP library (for NLP transformations).
In particular, spacy comes with pretrained language models which can be used for a variety of purposes, such as tokenization, entity recognition, and so forth. These models must be installed individually in your environment.

To install spacy and the english language model, you can use the following commands:

- pip install spacy
- python -m spacy download en_core_web_sm

You can find more information on spacy at https://spacy.io/usage/spacy-101

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

pd.set_option("display.max_colwidth", 200)

## The data

Each row of `contact_history.csv` is a contact between the agency and a debtor (an email, a phone call, a letter, and so forth). We only keep the emails. The text of an email is in the `email_body_snippet` column.

In [ ]:
contacts = pd.read_csv("../datasets/debt_collection/contact_history.csv")
emails = contacts[contacts["channel"] == "Email"].reset_index(drop=True)
print(emails.shape)
emails[["direction", "outcome", "email_subject", "email_body_snippet"]].head()

In [ ]:
emails["outcome"].value_counts()

In [ ]:
emails["direction"].value_counts()

Let's read a few of the emails:

In [ ]:
for text in emails["email_body_snippet"].sample(5, random_state=1):
    print(text)
    print("-" * 80)

**Question 1.** Who writes these emails? Read a few more of them. What do the emails of the agency have in common, and how are the emails of the debtors different?

## Tokenization

The first thing we need to do is to transform the emails into lists of tokens. A token can be defined as a string of contiguous characters between two spaces, or between a space and punctuation marks, but there are a lot of exceptions to this definition depending on the language used. Spacy NLP models have been trained to tokenize text for specific languages. Below, we see an example for English.

In [ ]:
# load the English language model of spacy, which we installed above
nlp = spacy.load("en_core_web_sm")

example = emails["email_body_snippet"].iloc[0]
print(example)
[token.text for token in nlp(example)]

Note that the spacy language model has been trained to recognize exceptions to the tokenization rules. For instance, in the below, "It's" is split into two tokens, while "N.Y." is treated as a single token.

In [ ]:
tokenization_example = nlp("It's cold in winter in N.Y.")
[token.text for token in tokenization_example]

One of the features of spacy is that the application of the nlp() function (i.e., the "language" object) does not only do tokenization, but applies a full pre-trained NLP pipeline to the text:

<div>
<img src="img/spacy_nlp_pipeline.png" width="600">
</div>

Spacy will first take the text and turn it into a list of tokens (a **doc** object), and then use a variety of pre-trained statistical models to annotate these tokens with additional information. For instance, the NER model in the pipeline attempts to identify tokens that refer to specific entities, such as people, organizations, dates or amounts of money. We call this **named entity recognition**, and it is a fundamental task in NLP. Below are the entities that spacy recognized in our example email (note that it's not perfect).

In [ ]:
for ent in nlp(example).ents:
    print(f"{ent.text!r:30} -> {ent.label_}")

In our application, however, we only need the tokenization; we are not interested in the other elements of the pipeline. Hence we will only run the rule-based tokenizer using nlp.tokenizer.pipe, which is a lot faster than running the statistical models.

## Data cleaning

Before tokenizing, we need to clean the emails a bit. They contain things that don't say anything about the topic of the email:

- many emails start with a "subject:" line. We remove it (the subject is also in its own column, `email_subject`)
- the account numbers, e.g. 11188d48. Every account has a different one, so they would only add noise
- dates, e.g. 2024-03-18, and the phone number of the agency

We remove these using **regular expressions**. We also lowercase everything, so that "Payment" and "payment" are counted as the same word.

In [ ]:
subject_re = re.compile(r"^subject:.*\n?")  # a subject line at the start
account_re = re.compile(r"\b[0-9a-f]{8}\b")  # 8 hexadecimal characters
date_re = re.compile(r"\d{4}-\d{2}-\d{2}")  # yyyy-mm-dd
phone_re = re.compile(r"0800-\d+")


def clean(text):
    text = text.lower()
    text = subject_re.sub("", text)
    text = account_re.sub(" ", text)
    text = date_re.sub(" ", text)
    text = phone_re.sub(" ", text)
    return text


print(example)
print("-" * 80)
print(clean(example))

Stop words (like "the", "and", "we"), punctuation, and numbers (here, the amounts) also don't encode semantic content and should be removed before doing topic modelling. Spacy comes with a built in list of stopwords:

In [ ]:
stopwords = nlp.Defaults.stop_words
print(len(stopwords))
sorted(stopwords)[:30]

In [ ]:
def tokenize(texts):
    tokens = []
    for doc in nlp.tokenizer.pipe(texts, batch_size=2000):
        tokens.append([
            token.text
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and not token.is_space
            and not token.like_num  # numbers, including amounts like 1952.92
            and len(token.text) > 1
        ])
    return tokens


email_tokens = tokenize(emails["email_body_snippet"].map(clean))
email_tokens[:3]

Note that `email_tokens` is not a list of doc objects anymore: it is now a list of lists of strings, because we don't need the spacy tokens from now on.

## The documents x words matrix

Now that we have tokenized and cleaned the emails, we are going to use sklearn CountVectorizer to create a big matrix of dimension N x M, where N are the individual documents (the emails), and the columns M of the matrix are the individual words that occur in the *corpus* (i.e., in the whole list of documents). The values of the matrix cells V_ij will be the occurrence counts of word j in document i.

We use two arguments to decide which words to keep:

- min_df=5 removes the words that occur in fewer than 5 emails. These are typically typos or very specific details, and they are too rare to say anything about a topic
- max_df=0.5 removes the words that occur in more than half of the emails. Words like "account" occur almost everywhere in this corpus, so they don't help to tell emails apart

In [ ]:
# we need to pass a dummy function to the tokenizer and preprocessor parameters of count vectorizer
# because we already calculated our tokens
def identity(doc):
    return doc


count_vectorizer = CountVectorizer(
    tokenizer=identity,
    preprocessor=identity,
    token_pattern=None,  # needed because we use our own tokenizer
    min_df=5,
    max_df=0.5,
)
frequency_matrix = count_vectorizer.fit_transform(email_tokens)
frequency_matrix

Note that sklearn stores the frequency matrix as a sparse matrix, which is more efficient for 0 inflated matrices like the one above. Below we look at the vocabulary, and at the words that occur in the most emails (the ones above 0.5 were removed by max_df).

In [ ]:
vocabulary = count_vectorizer.get_feature_names_out()
print(len(vocabulary))
vocabulary[:50]

In [ ]:
# in what share of the emails does each word occur? (binary=True counts presence, not occurrences)
presence = CountVectorizer(
    tokenizer=identity, preprocessor=identity, token_pattern=None, binary=True
).fit(email_tokens)
share = np.asarray(presence.transform(email_tokens).mean(axis=0)).ravel()
document_frequency = pd.Series(share, index=presence.get_feature_names_out())
document_frequency.sort_values(ascending=False).head(15)  # those above 0.5 were dropped

Note also that words like "pay", "paying" and "payment" are still considered as separate words. This is not perfect, because their semantic content is basically the same. To fix this one can apply **lemmatization**, which identifies the underlying lemma of each word (in this case, "pay"). We will do this later in the notebook, for the replies of the debtors.

**Question 2.** Set max_df=1.0 (i.e., keep all the words) and rerun the cells until the topics. What happens to the topics?

Next, we apply the tf-idf transformer to turn the frequency matrix into a tf-idf matrix. Tf-idf stands for: *term frequency inverse document frequency*. It is a statistical measure of how relevant a word is to a given document, in the context of a collection of documents.

It is calculated as: **frequency of a word in a document * inverse document frequency of the word across all documents**.

The inverse document frequency encodes how rare a word is in the whole dataset. The closer to 0 the inverse document frequency is, the more common the word is. It is calculated by taking the total number of documents, dividing it by the number of documents containing the word, and then taking the logarithm.

The idea is that if a word has a high idf, then it is a rare word in the corpus, and hence it is likely to carry more of the semantic content of the document than very common words. E.g., in our emails a word like "dispute" says a lot more about an email than a word like "contact".

<div>
<img src="img/tf_idf.jpeg" width="500">
</div>

In [ ]:
tfidf_transformer = TfidfTransformer()
tfidf_matrix = tfidf_transformer.fit_transform(frequency_matrix)
tfidf_matrix

## Topic modelling with NMF

We now use the NMF model from sklearn to decompose our tf-idf matrix V into two matrices: a document x topics matrix W, and a topics x words matrix H. Thus, the model finds two matrices W, H, such that V ~ WH.

Note the sign ~ in the equation above, and not the sign =. This is because the two matrices W, H will not return the original tf-idf matrix when multiplied together; rather, they will return a new matrix that approximates it with an error.

The topics are **hidden** (or latent): they are not in the data, and nobody told the model what they are. The model finds them by looking at which words tend to occur together in the same emails. And since all the entries of W and H are non negative, an email can only be built by *adding* topics together, and a topic by adding words together. This is why NMF topics are often quite easy to interpret.

One important caveat of this model is that the number of topics to be discovered (the n_components parameter in the NMF model), i.e., the number of columns of W and the number of rows of H, is a **hyperparameter**; i.e., it needs to be specified when the model is instantiated. We use 8 here, and come back to this later.

In [ ]:
# we use 8 as the n_components here
model = NMF(n_components=8, init="nndsvd", random_state=0, max_iter=500)

# fit the model, and get the W matrix (emails x topics)
W = model.fit_transform(tfidf_matrix)
# the model.components_ matrix is the H matrix (topics x words)
H = model.components_
print(W.shape, H.shape)

Let's look at the two matrices. The model.components_ matrix is the H matrix: it has a row for each topic, and a column for each word in the vocabulary. Below are some of its columns:

In [ ]:
H_df = pd.DataFrame(H, columns=vocabulary, index=[f"Topic {i}" for i in range(model.n_components)])
H_df[["arrangement", "court", "dispute", "reminder", "sorry", "thank"]].round(3)

You can see that every word has a weight in every topic, but most of them are 0 or close to 0. E.g., "court" is important for topics 0 and 3, and "arrangement" for topic 1. The words with the highest weights define the topic.

The W matrix instead has a row for each email, and a column for each topic:

In [ ]:
W_df = pd.DataFrame(W, columns=[f"Topic {i}" for i in range(model.n_components)])
W_df.head().round(3)

Most emails only have weight on a few topics. To understand what the topics are about, we print for each topic (each row of H) the words with the highest weights:

In [ ]:
def get_topics(model, vocabulary, n_top_words):
    '''This function takes a fitted NMF factorization model, its vocabulary, and a n_top_words parameter.
    It then produces a dataframe where the columns are the topics that have been learned, and the
    rows are the top words that define the topic, ranked by their coefficients.
    '''
    topics = {}
    for i, weights in enumerate(model.components_):
        top_words = weights.argsort()[::-1][:n_top_words]
        topics[f"Topic {i}"] = vocabulary[top_words]
    return pd.DataFrame(topics)


get_topics(model, vocabulary, 15)

We see some coherence in the topics that the NMF model has identified. In particular:

- Topic 0 has "court", "enforcement", "costs" and "judgment", so it looks like it's about the consequences of not paying
- Topic 1 is about confirming a payment arrangement, in a friendly tone ("thank", "appreciate", "pleased")
- Topic 2 seems to be a friendly reminder about the outstanding balance
- Topic 3 is the final notice before legal action
- Topics 4 and 6 are about the debt being referred to a collection agency or department. Note however that they are also defined by the way the letter is written ("dear", "holder", "regards", "faithfully", "team"), so they are as much about *how* the agency writes as about *what* it writes
- Topic 5 is quite different from all the others: "need", "got", "know", "want", "charges", "owe", "breakdown". These look like emails written by the debtors themselves
- Topic 7 is about answering a question of the debtor and discussing the options

**Question 3.** Do you agree with the names above? Which topics are about *what* the email says, and which ones about *how* it says it?

## Checking the topics

We don't have labels for the topics, but we do have some columns that should be related to them. We can use the model to identify the main topic of each email, i.e. the topic with the highest weight in its row of W.

In [ ]:
emails["main_topic"] = W.argmax(axis=1)
emails["main_topic"].value_counts().sort_index()

Let's read a few emails of topic 5, to check that they are really written by debtors:

In [ ]:
for text in emails.loc[emails["main_topic"] == 5, "email_body_snippet"].sample(4, random_state=0):
    print(text)
    print("-" * 80)

Now we compare the topics with the `outcome` and the `email_subject` columns:

In [ ]:
pd.crosstab(emails["main_topic"], emails["outcome"])

In [ ]:
pd.crosstab(emails["main_topic"], emails["email_subject"])

Topic 5 is almost exactly the emails with outcome "Email replied" (6691 of its 6931 emails). So the model found the replies of the debtors without being told who wrote what, just because the debtors use different words than the templates of the agency.

The other topics line up quite well with the subject of the email, even though the model never saw it (we removed the subject line during the cleaning). E.g., topic 1 goes with "Your payment arrangement" and "Confirmation of your payment plan", topic 3 with "Urgent: Final notice before legal action", and topic 7 with "Your recent query". The replies (topic 5) are spread over all the subjects, which makes sense since a reply keeps the subject of the email it answers. This is a good sign that the topics are picking up real structure in the data.

In [ ]:
pd.crosstab(emails["main_topic"], emails["direction"])

**Question 4.** If topic 5 contains the emails of the debtors, they should be inbound emails. What does the `direction` column say? What does this tell you about this column, and how would you check it?

## Emails as mixtures of topics

Until now we assigned each email to one main topic. But the W matrix tells us more than that: row i of W gives the weight of each topic in email i. If we divide each row by its sum, we get the share of each topic in the email.

In [ ]:
topic_names = [
    "0 consequences", "1 arrangement", "2 reminder", "3 final notice",
    "4 agency referral", "5 debtor reply", "6 department referral", "7 query",
]
shares = pd.DataFrame(W / W.sum(axis=1, keepdims=True), columns=topic_names)
shares["main topic share"] = shares.max(axis=1)
shares["main topic share"].describe()

For half of the emails, the main topic accounts for less than 53% of the weight, and for a quarter of them for less than 43%. So most emails are really a mix of several topics, and by only looking at the main topic we throw this information away. Below are two emails where three or more topics have at least 20% of the weight:

In [ ]:
# emails where three topics each have at least 20% of the weight
mixed = shares.index[(shares[topic_names] >= 0.2).sum(axis=1) >= 3][:2]
for i in mixed:
    print(emails.loc[i, "email_body_snippet"])
    print(shares.loc[i, topic_names].round(2)[lambda x: x > 0].sort_values(ascending=False).to_string())
    print("-" * 80)

Both emails are final demands, and the model describes each of them as a mix of several escalation topics rather than as a single topic. The first one is mostly a referral to the collections department, plus the final notice and the threat of enforcement. The second one does a bit of everything: it mentions the agency, threatens court and costs, is signed by the department, and calls itself a final opportunity.

Note that this is different from a clustering method like k-means, where each email would belong to exactly one cluster.

**Question 5.** Which topics appear together most often in the same email? (Hint: for each email, find the two topics with the largest shares, and count the pairs.) What does this tell you about the way the agency writes its emails?

## Choosing the number of topics

As said above, the number of topics is a hyperparameter. One way to choose it is to look at the reconstruction error, i.e. how far WH is from V. More topics will always give a lower error, so we look for the point where adding more topics doesn't help much anymore (the "elbow").

In [ ]:
ks = range(2, 17, 2)
errors = []
for k in ks:
    nmf = NMF(n_components=k, init="nndsvd", random_state=0, max_iter=500).fit(tfidf_matrix)
    errors.append(nmf.reconstruction_err_)

plt.plot(list(ks), errors, marker="o")
plt.xlabel("number of topics")
plt.ylabel("reconstruction error")
plt.show()

In our case the error goes down slowly and steadily, and there is no clear elbow. This happens quite often with text data. So in practice we choose a number of topics that we can still interpret, and that is useful for the question we are trying to answer.

**Question 6.** Fit the model with 4 and with 12 topics. Which topics merge, and which ones split? How many topics would you use in a report for the management of the agency, and why?

## Topic modelling on the replies of the debtors

Most emails are written by the agency, but for the agency the interesting ones are the replies of the debtors, since those are the emails that someone has to read and act on. So let's keep only the emails with outcome "Email replied", and do topic modelling on those.

This time we also apply lemmatization, so that "paying", "paid" and "pays" are all counted as "pay". Lemmatization needs the statistical models of spacy (the lemma of a word depends on its part of speech), so we use nlp.pipe instead of only the tokenizer. We disable the parser and the NER, which we don't need, to make it faster.

In [ ]:
replies = emails[emails["outcome"] == "Email replied"].reset_index(drop=True)
print(len(replies))

reply_lemmas = []
for doc in nlp.pipe(replies["email_body_snippet"].map(clean), batch_size=500, disable=["parser", "ner"]):
    reply_lemmas.append([
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
        and not token.like_num
        and len(token.text) > 1
    ])
reply_lemmas[:3]

In [ ]:
reply_vectorizer = CountVectorizer(
    tokenizer=identity, preprocessor=identity, token_pattern=None, min_df=5, max_df=0.5
)
reply_tfidf = TfidfTransformer().fit_transform(reply_vectorizer.fit_transform(reply_lemmas))
reply_vocabulary = reply_vectorizer.get_feature_names_out()

reply_model = NMF(n_components=6, init="nndsvd", random_state=0, max_iter=500)
reply_W = reply_model.fit_transform(reply_tfidf)
get_topics(reply_model, reply_vocabulary, 15)

In [ ]:
replies["main_topic"] = reply_W.argmax(axis=1)
for topic in range(reply_model.n_components):
    print(f"===== Topic {topic} ({(replies['main_topic'] == topic).sum()} replies)")
    for text in replies.loc[replies["main_topic"] == topic, "email_body_snippet"].sample(2, random_state=0):
        print(text)
        print()

With the lemmatization, and only the replies, the topics are now about what the debtors want:

- Topic 0 is a formal commitment to pay, often with an apology ("settle", "confirm", "apologize", "patience")
- Topic 1 seems to be the informal replies ("know", "sort", "sorry"), usually a promise to pay and sometimes a question
- Topic 2 are requests for information: a statement, a breakdown of the balance, the original charges, interest and fees
- Topic 3 is about disputing the debt ("dispute", "wrong", "stop", "evidence", "proof")
- Topic 4 are reactions to a final notice: the debtor understands the deadline and the consequences, and often pays right away
- Topic 5 is a formal acknowledgement of the balance, often with a proposal for an arrangement

Note the first example of topic 3: "Not disputing any of it". It ends up in the dispute topic because it contains the word "disputing". This is a limitation of bag of words models: they ignore the order of the words, so they can't see the negation. A transformer, like the one of the transformers tutorial, looks at the whole sentence and is much less likely to make this mistake.

**Question 7.** The agency wants to route the replies automatically: some can be answered with a standard message, while others need a person. Based on the topics above, which replies would you send to a person, and why? What could go wrong if you only route on the main topic?